In [1]:
import pandas as pd
import json

# Load the CSV
amp_df = pd.read_csv('../../data/raw/amp_mic_activities.csv')

In [2]:
display(amp_df)

,sequence,target_activity_name,activity
0,RRXXRF,Bacillus subtilis PY22,4.277080
1,RRXXRF,Escherichia coli DH5alpha,17.108320
2,KVvvKWVvKvVK,Staphylococcus aureus ATCC 6538P,165.228990
3,KVvvKWVvKvVK,Bacillus subtilis ATCC 6051,165.228990
4,KVvvKWVvKvVK,Pseudomonas aeruginosa ATCC 27853,165.228990
...,...,...,...
114875,RxXxR,Staphylococcus aureus ATCC 43300,4.237459
114876,RxXxR,Staphylococcus epidermidis ATCC 35984,4.237459
114877,RxXxR,Enterococcus faecalis ATCC 29212,16.949835
114878,RxXxR,Enterococcus faecium ATCC 700221,8.474917


In [3]:
!conda install certifi openssl

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - defaults
Platform: osx-arm64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.9.1
    latest version: 26.5.0

Please update conda by running

    $ conda update -n base -c defaults conda



# All requested packages already installed.



In [4]:
!pip install --upgrade certifi

  Attempting uninstall: certifi
    Found existing installation: certifi 2026.4.22
    Uninstalling certifi-2026.4.22:
      Successfully uninstalled certifi-2026.4.22


In [5]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [6]:
from ete3 import NCBITaxa

ncbi = NCBITaxa()

In [7]:
# ncbi.update_taxonomy_database()

In [8]:
import re

def get_taxonomic_lineage(name):
    empty_lineage = {'Phylum': 'Unknown', 'Class': 'Unknown', 'Order': 'Unknown', 'Family': 'Unknown', 'Genus': 'Unknown'}
    
    # 1. Clean the name: Remove "ATCC xxxx", "strain", and extra codes
    # This regex helps keep only the Genus and Species
    clean_name = re.sub(r'(ATCC|strain|MTCC|v\.|subsp\.).*', '', name, flags=re.IGNORECASE).strip()
    
    try:
        # Try to find the taxid
        taxids = ncbi.get_name_translator([clean_name])
        
        if not taxids:
            # Fallback: just take the first two words (Genus species)
            words = clean_name.split()
            if len(words) >= 2:
                clean_name = f"{words[0]} {words[1]}"
                taxids = ncbi.get_name_translator([clean_name])
        
        if not taxids:
            return empty_lineage
            
        taxid = list(taxids.values())[0][0]
        lineage = ncbi.get_lineage(taxid)
        ranks = ncbi.get_rank(lineage)
        names = ncbi.get_taxid_translator(lineage)
        
        lineage_dict = empty_lineage.copy()
        for rank_id, rank_name in ranks.items():
            if rank_name in lineage_dict:
                lineage_dict[rank_name] = names[rank_id]
        
        return lineage_dict

    except:
        return empty_lineage

In [11]:
# Create a manual mapping for the most common AMP targets
common_taxonomy = {
    'Escherichia coli': {'Phylum': 'Pseudomonadota', 'Class': 'Gammaproteobacteria', 'Order': 'Enterobacterales', 'Family': 'Enterobacteriaceae', 'Genus': 'Escherichia'},
    'Bacillus subtilis': {'Phylum': 'Bacillota', 'Class': 'Bacilli', 'Order': 'Bacillales', 'Family': 'Bacillaceae', 'Genus': 'Bacillus'},
    'Staphylococcus aureus': {'Phylum': 'Bacillota', 'Class': 'Bacilli', 'Order': 'Bacillales', 'Family': 'Staphylococcaceae', 'Genus': 'Staphylococcus'},
    'Pseudomonas aeruginosa': {'Phylum': 'Pseudomonadota', 'Class': 'Gammaproteobacteria', 'Order': 'Pseudomonadales', 'Family': 'Pseudomonadaceae', 'Genus': 'Pseudomonas'}
}

def get_lineage_v2(name):
    # Standardize name to Genus species
    words = name.split()
    if len(words) >= 2:
        short_name = f"{words[0]} {words[1]}"
        if short_name in common_taxonomy:
            return common_taxonomy[short_name]
            
    # If not in our manual map, try ete3 one last time
    try:
        taxids = ncbi.get_name_translator([name])
        
        if not taxids:
            # Fallback: just take the first two words (Genus species)
            words = clean_name.split()
            if len(words) >= 2:
                clean_name = f"{words[0]} {words[1]}"
                taxids = ncbi.get_name_translator([clean_name])
        
        if not taxids:
            return empty_lineage
            
        taxid = list(taxids.values())[0][0]
        lineage = ncbi.get_lineage(taxid)
        ranks = ncbi.get_rank(lineage)
        names = ncbi.get_taxid_translator(lineage)
        
        lineage_dict = empty_lineage.copy()
        for rank_id, rank_name in ranks.items():
            if rank_name in lineage_dict:
                lineage_dict[rank_name] = names[rank_id]
        
        return lineage_dict

    except:
        return {'Phylum': 'Unknown', 'Class': 'Unknown', 'Order': 'Unknown', 'Family': 'Unknown', 'Genus': 'Unknown'}

In [13]:
# Test a known bacterium
test_name = "Escherichia coli"
print(f"Testing {test_name}:", get_lineage_v2(test_name))

# Test one of your specific strings
test_name_2 = "Bacillus subtilis PY22"
print(f"Testing {test_name_2}:", get_lineage_v2(test_name_2))

Testing Listeria monocytogenes EGD: {'Phylum': 'Unknown', 'Class': 'Unknown', 'Order': 'Unknown', 'Family': 'Unknown', 'Genus': 'Unknown'}
Testing Bacillus subtilis PY22: {'Phylum': 'Bacillota', 'Class': 'Bacilli', 'Order': 'Bacillales', 'Family': 'Bacillaceae', 'Genus': 'Bacillus'}


In [36]:
# 1. Get unique bacteria names to optimize speed
unique_bacteria = amp_df['target_activity_name'].unique()

# 2. Fetch lineages for unique elements
lineage_data = [get_lineage_v2(b) for b in unique_bacteria]

# # 3. Create a lookup dataframe
taxonomy_lookup = pd.DataFrame(lineage_data)
taxonomy_lookup['target_activity_name'] = unique_bacteria

# # 4. Merge the taxonomy back into your main dataset
amp_df = amp_df.merge(taxonomy_lookup, on='target_activity_name', how='left')

In [26]:
amp_df

,sequence,target_activity_name,activity,Phylum_x,Class_x,Order_x,Family_x,Genus_x,Phylum_y,Class_y,Order_y,Family_y,Genus_y
0,RRXXRF,Bacillus subtilis PY22,4.277080,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
1,RRXXRF,Escherichia coli DH5alpha,17.108320,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
2,KVvvKWVvKvVK,Staphylococcus aureus ATCC 6538P,165.228990,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
3,KVvvKWVvKvVK,Bacillus subtilis ATCC 6051,165.228990,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
4,KVvvKWVvKvVK,Pseudomonas aeruginosa ATCC 27853,165.228990,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...
114875,RxXxR,Staphylococcus aureus ATCC 43300,4.237459,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
114876,RxXxR,Staphylococcus epidermidis ATCC 35984,4.237459,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
114877,RxXxR,Enterococcus faecalis ATCC 29212,16.949835,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
114878,RxXxR,Enterococcus faecium ATCC 700221,8.474917,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown


In [27]:
# Select the taxonomic columns to vectorize
rank_columns = ['Phylum', 'Class', 'Order', 'Family', 'Genus']

# Convert to fixed-length one-hot encoded vector columns
bacteria_vectors = pd.get_dummies(amp_df[rank_columns], prefix=rank_columns, dtype=int)

# Concatenate back to the main DataFrame
final_df = pd.concat([amp_df, bacteria_vectors], axis=1)

# Drop the text taxonomic columns to clean up
final_df = final_df.drop(columns=rank_columns)

KeyError: "None of [Index(['Phylum', 'Class', 'Order', 'Family', 'Genus'], dtype='object')] are in the [columns]"

In [18]:
display(final_df)

,sequence,target_activity_name,activity,Phylum_Unknown,Class_Unknown,Order_Unknown,Family_Unknown,Genus_Unknown
0,RRXXRF,Bacillus subtilis PY22,4.277080,1,1,1,1,1
1,RRXXRF,Escherichia coli DH5alpha,17.108320,1,1,1,1,1
2,KVvvKWVvKvVK,Staphylococcus aureus ATCC 6538P,165.228990,1,1,1,1,1
3,KVvvKWVvKvVK,Bacillus subtilis ATCC 6051,165.228990,1,1,1,1,1
4,KVvvKWVvKvVK,Pseudomonas aeruginosa ATCC 27853,165.228990,1,1,1,1,1
...,...,...,...,...,...,...,...,...
114875,RxXxR,Staphylococcus aureus ATCC 43300,4.237459,1,1,1,1,1
114876,RxXxR,Staphylococcus epidermidis ATCC 35984,4.237459,1,1,1,1,1
114877,RxXxR,Enterococcus faecalis ATCC 29212,16.949835,1,1,1,1,1
114878,RxXxR,Enterococcus faecium ATCC 700221,8.474917,1,1,1,1,1


In [23]:
final_df['Genus_Unknown'].unique()

array([1])